# Perbandingan Performa Group-By: CSV vs Lakehouse (Iceberg)

Notebook ini membandingkan waktu eksekusi query group-by sederhana:
1) Baca langsung dari CSV (s3a://tpch-raw/lineitem.tbl)
2) Baca dari Iceberg (local.tpch.lineitem)

Prasyarat: Docker stack berjalan, CSV sudah di-upload ke MinIO, dan tabel Iceberg sudah dibuat (lihat load_and_read.ipynb).

In [1]:
import os
from pyspark.sql import SparkSession

JARS = ",".join([
    "/home/jovyan/extra-jars/iceberg-spark-runtime-3.5_2.12-1.4.3.jar",
    "/home/jovyan/extra-jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/extra-jars/aws-java-sdk-bundle-1.12.262.jar",
])
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {JARS} pyspark-shell"

spark = (
    SparkSession.builder
    .appName("compare-csv-vs-lakehouse")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://lakehouse/")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
import time
from pyspark.sql import functions as F

LINEITEM_SCHEMA = (
    "l_orderkey LONG, l_partkey LONG, l_suppkey LONG, l_linenumber INT, "
    "l_quantity DECIMAL(15,2), l_extendedprice DECIMAL(15,2), l_discount DECIMAL(15,2), "
    "l_tax DECIMAL(15,2), l_returnflag STRING, l_linestatus STRING, l_shipdate DATE, "
    "l_commitdate DATE, l_receiptdate DATE, l_shipinstruct STRING, l_shipmode STRING, "
    "l_comment STRING, _trailing STRING"
)

def groupby_query(df):
    return (
        df.groupBy("l_returnflag", "l_linestatus")
        .agg(
            F.sum("l_quantity").alias("sum_qty"),
            F.sum("l_extendedprice").alias("sum_base_price"),
            F.avg("l_discount").alias("avg_disc"),
            F.count("*").alias("count_order"),
        )
        .orderBy("l_returnflag", "l_linestatus")
    )

def timed_collect(label, df):
    start = time.perf_counter()
    rows = df.collect()
    elapsed = time.perf_counter() - start
    print(f"{label} - elapsed: {elapsed:.3f}s, rows: {len(rows)}")
    return elapsed, rows

In [3]:
# CSV langsung dari MinIO
csv_df = (
    spark.read
    .option("delimiter", "|")
    .option("header", "false")
    .schema(LINEITEM_SCHEMA)
    .csv("s3a://tpch-raw/lineitem.tbl")
    .drop("_trailing")
)
csv_elapsed, csv_rows = timed_collect("CSV group-by", groupby_query(csv_df))
print("CSV result rows:")
for r in csv_rows:
    print(r)

# Iceberg (Lakehouse)
iceberg_df = spark.table("local.tpch.lineitem")
iceberg_elapsed, iceberg_rows = timed_collect("Iceberg group-by", groupby_query(iceberg_df))
print("Iceberg result rows:")
for r in iceberg_rows:
    print(r)

if iceberg_elapsed > 0:
    ratio = csv_elapsed / iceberg_elapsed
    print(f"Speedup (CSV / Iceberg): {ratio:.2f}x")

CSV group-by - elapsed: 20.816s, rows: 4
CSV result rows:
Row(l_returnflag='A', l_linestatus='F', sum_qty=Decimal('377518399.00'), sum_base_price=Decimal('566065727797.25'), avg_disc=Decimal('0.050007'), count_order=14804077)
Row(l_returnflag='N', l_linestatus='F', sum_qty=Decimal('9851614.00'), sum_base_price=Decimal('14767438399.17'), avg_disc=Decimal('0.049973'), count_order=385998)
Row(l_returnflag='N', l_linestatus='O', sum_qty=Decimal('764635193.00'), sum_base_price=Decimal('1146548935600.94'), avg_disc=Decimal('0.050001'), count_order=29987794)
Row(l_returnflag='R', l_linestatus='F', sum_qty=Decimal('377732830.00'), sum_base_price=Decimal('566431054976.00'), avg_disc=Decimal('0.049997'), count_order=14808183)
Iceberg group-by - elapsed: 5.034s, rows: 4
Iceberg result rows:
Row(l_returnflag='A', l_linestatus='F', sum_qty=Decimal('377518399.00'), sum_base_price=Decimal('566065727797.25'), avg_disc=Decimal('0.050007'), count_order=14804077)
Row(l_returnflag='N', l_linestatus='F', s